In [1]:
import tensorflow as tf
import numpy as np
import os
import tempfile
import zipfile

In [2]:
model_path = "models/base_models/cassava_edge_baseline_model.h5"

model = tf.keras.models.load_model(model_path)
print("Model loaded successfully.")
model.summary()

Model loaded successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 4)           │             112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 4)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 8)           │             296 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 8)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 12)          │             876 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 12)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 432)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 16)                  │           6,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │              85 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,299 (32.42 KB)

 Trainable params: 8,297 (32.41 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [3]:
MODEL_TFLITE_PATH = "models/optimized_models/cassava_edge_model_dynamic.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model_dynamic = converter.convert()

with open(MODEL_TFLITE_PATH, "wb") as f:
    f.write(tflite_model_dynamic)

print("Dynamic quantized model saved successfully.")

INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmpkkwfu4v3\assets


INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmpkkwfu4v3\assets


Saved artifact at 'C:\Users\nickh\AppData\Local\Temp\tmpkkwfu4v3'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1945281560064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945281680656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286246240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286245184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286337632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286337456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286462272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286461920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286543840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1945286543664: TensorSpec(shape=(), dtype=tf.resource, name=None)
Dynamic qu

In [4]:
def get_gzipped_model_size(file):
    temp = tempfile.NamedTemporaryFile(suffix=".zip", delete=False)
    temp.close()

    with zipfile.ZipFile(temp.name, "w", compression=zipfile.ZIP_DEFLATED) as f:
        f.write(file)

    size = os.path.getsize(temp.name)
    os.remove(temp.name)
    return size

fp_size = get_gzipped_model_size(model_path)
quant_size = get_gzipped_model_size(MODEL_TFLITE_PATH)

print("Size of Full Precision Model: {} Bytes".format(fp_size))
print("Size of Quantized Model: {} Bytes".format(quant_size))
print("Size reduction factor: {:.2f}x".format(fp_size / quant_size))

Size of Full Precision Model: 94531 Bytes
Size of Quantized Model: 14107 Bytes
Size reduction factor: 6.70x


In [5]:
# Load test data
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# Load TFLite model
tflite_interpreter = tf.lite.Interpreter(model_path=MODEL_TFLITE_PATH)
tflite_interpreter.allocate_tensors()

input_details = tflite_interpreter.get_input_details()
output_details = tflite_interpreter.get_output_details()

predictions = np.zeros(len(X_test), dtype=int)

for i in range(len(X_test)):
    val_batch = X_test[i]

    # Dynamic quantization uses float32 input, so no manual quantization scaling here
    val_batch = np.expand_dims(val_batch, axis=0).astype(input_details[0]["dtype"])

    tflite_interpreter.set_tensor(input_details[0]['index'], val_batch)
    tflite_interpreter.invoke()

    output = tflite_interpreter.get_tensor(output_details[0]['index'])
    predictions[i] = output.argmax()

correct = 0
for i in range(len(predictions)):
    if predictions[i] == y_test[i]:
        correct += 1

accuracy_score = correct / len(predictions)

# Load full precision model
full_precision_model = tf.keras.models.load_model(model_path, compile=False)
full_precision_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

score = full_precision_model.evaluate(X_test, y_test, verbose=0)

print("Accuracy of quantized model is {}%".format(accuracy_score * 100))
print("Compared to float32 accuracy of {}%".format(score[1] * 100))
print("We have a change of {}%".format((accuracy_score - score[1]) * 100))

C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Accuracy of quantized model is 65.23364485981308%
Compared to float32 accuracy of 65.21028280258179%
We have a change of 0.023362057231302025%
